<a href="https://colab.research.google.com/github/yamazaki-riko/python_learning/blob/koshien_google-colab/koshien_bt_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import sqlite3
import pandas as pd
import numpy as np
from scipy.optimize import minimize

drive.mount('/content/drive')
conn = sqlite3.connect("/content/drive/MyDrive/koshien/koshien.db")
print("✅ DB接続完了")

Mounted at /content/drive
✅ DB接続完了


In [2]:
df_games = pd.read_sql("""
    SELECT year, winner_school, loser_school
    FROM tmp_tournament_games
    WHERE winner_score IS NOT NULL
""", conn)

print(f"試合数: {len(df_games)}試合")
display(df_games.head())

試合数: 207試合


,year,winner_school,loser_school
0,2022,国学院栃木,日大三島
1,2022,明豊,樹徳
2,2022,一関学院,京都国際
3,2022,八戸学院光星,愛工大名電
4,2022,愛工大名電,星稜


In [3]:
def calc_bradley_terry(df):
    teams = sorted(set(df["winner_school"]) | set(df["loser_school"]))
    idx   = {t: i for i, t in enumerate(teams)}
    n     = len(teams)

    def neg_log_likelihood(params):
        strength = np.exp(params)
        total = 0.0
        for _, row in df.iterrows():
            w = idx[row["winner_school"]]
            l = idx[row["loser_school"]]
            total -= np.log(strength[w] / (strength[w] + strength[l]))
        return total

    result = minimize(
        neg_log_likelihood,
        x0=np.zeros(n),
        method="L-BFGS-B"
    )

    strength = np.exp(result.x)
    df_result = pd.DataFrame({
        "school":   teams,
        "strength": strength
    }).sort_values("strength", ascending=False).reset_index(drop=True)
    df_result["rank"] = df_result.index + 1
    return df_result

print("✅ 関数の定義完了")

✅ 関数の定義完了


In [4]:
# 2022〜2023年のデータで強さを計算
df_train = df_games[df_games["year"] <= 2023]
df_bt    = calc_bradley_terry(df_train)

# 2024年の出場校を取得
df_2024 = pd.read_sql("""
    SELECT school_name, district
    FROM tmp_koshien_appearances_summer
    WHERE year = 2024
""", conn)

# BTモデルの結果と結合
df_pred = df_2024.merge(
    df_bt[["school", "strength", "rank"]],
    left_on  = "school_name",
    right_on = "school",
    how      = "left"
)

# 強さでソートしてポイントを割り当て
df_pred = df_pred.sort_values("strength", ascending=False).reset_index(drop=True)
df_pred["bt_rank"]   = df_pred.index + 1
df_pred["bt_points"] = range(49, 0, -1)

print("🏆 2024年 BTモデル予測ランキング（上位20校）")
display(df_pred[["bt_rank","school_name","district","bt_points"]].head(20))


🏆 2024年 BTモデル予測ランキング（上位20校）


,bt_rank,school_name,district,bt_points
0,1,愛工大名電,愛知,49
1,2,神村学園,鹿児島,48
2,3,富山商,富山,47
3,4,聖光学院,福島,46
4,5,花巻東,岩手,45
5,6,大阪桐蔭,大阪,44
6,7,広陵,広島,43
7,8,大垣日大,岐阜,42
8,9,北陸,福井,41
9,10,上田西,長野,40


In [5]:
# 2024年の実際の勝利数を集計
df_2024_actual = pd.read_sql("""
    SELECT winner_school, COUNT(*) AS wins
    FROM tmp_tournament_games
    WHERE year = 2024
    GROUP BY winner_school
    ORDER BY wins DESC
""", conn)

print("2024年 実際の勝利数ランキング:")
display(df_2024_actual)

# BTモデルの予測と実際の勝利数を比較
df_compare = df_pred[["bt_rank","school_name","bt_points"]].merge(
    df_2024_actual,
    left_on  = "school_name",
    right_on = "winner_school",
    how      = "left"
).fillna({"wins": 0})

df_compare["wins"] = df_compare["wins"].astype(int)

print("\n📊 予測 vs 実際の比較:")
display(df_compare[["bt_rank","school_name","bt_points","wins"]])

2024年 実際の勝利数ランキング:


,winner_school,wins
0,京都国際,6
1,関東第一,4
2,青森山田,3
3,神村学園,3
4,早稲田実,3
5,滋賀学園,2
6,東海大相模,2
7,岡山学芸館,2
8,大阪桐蔭,2
9,大社,2



📊 予測 vs 実際の比較:


,bt_rank,school_name,bt_points,wins
0,1,愛工大名電,49,0
1,2,神村学園,48,3
2,3,富山商,47,0
3,4,聖光学院,46,1
4,5,花巻東,45,0
5,6,大阪桐蔭,44,2
6,7,広陵,43,1
7,8,大垣日大,42,0
8,9,北陸,41,0
9,10,上田西,40,0
